In [ ]:
#train
# Install necessary libraries if not already installed
# pip install transformers datasets

# Import necessary libraries
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset

# Load GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Add a padding token to the tokenizer
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Resize the model's token embeddings to account for the new token
model.resize_token_embeddings(len(tokenizer))

# Load the CSV file into a Hugging Face dataset
dataset = load_dataset('csv', data_files='sample.csv')['train']

# Tokenize the dataset
def tokenize_function(examples):
    # Tokenize the text and add the labels for language modeling
    encodings = tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)
    encodings['labels'] = encodings['input_ids']  # Set labels for language modeling
    return encodings

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./gpt2finetuned",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_dir='./logs',
    logging_steps=10,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Fine-tune the model
trainer.train()

# Save the fine-tuned model
trainer.save_model("./gpt2finetuned")


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset

# Load GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Add a padding token to the tokenizer
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Resize the model's token embeddings to account for the new token
model.resize_token_embeddings(len(tokenizer))

# Load the CSV file into a Hugging Face dataset
dataset = load_dataset('csv', data_files='sample.csv')['train']

# Tokenize the dataset
def tokenize_function(examples):
    encodings = tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)
    encodings['labels'] = encodings['input_ids']
    return encodings

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./gpt2finetuned",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_dir='./logs',
    logging_steps=10,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Fine-tune the model
trainer.train()

# Save the fine-tuned model and tokenizer
trainer.save_model("./gpt2finetuned")
tokenizer.save_pretrained("./gpt2finetuned")


C:\Users\CUI\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [19]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the fine-tuned model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('./gpt2finetuned')
model = GPT2LMHeadModel.from_pretrained('./gpt2finetuned')
import torch

def generate_text(prompt, max_length=50, num_return_sequences=1):
    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors='pt')
    
    # Generate text
    outputs = model.generate(
        inputs['input_ids'],
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        no_repeat_ngram_size=2,  # To prevent repetition
        top_k=50,                # Top-k sampling
        top_p=0.95                # Top-p sampling (nucleus sampling)
    )
    
    # Decode the generated text
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_texts

# Example usage
prompt = "Amin"
generated_texts = generate_text(prompt)
print(generated_texts)



OSError: Can't load tokenizer for './gpt2finetuned'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure './gpt2finetuned' is the correct path to a directory containing all relevant files for a GPT2Tokenizer tokenizer.